# AI-Driven Online Subjective Answer Evaluator: Training & Evaluation

This notebook provides the tools to train a custom SBERT model and evaluate student answers against model answers using a hybrid Semantic + Keyword approach.

### 🔗 Original Colab Links
- [Evaluation Logic](https://colab.research.google.com/drive/19dXPNIJB937pMPU2ltcKVE8ryylR-C06)
- [OCR Extraction](https://colab.research.google.com/drive/1y85Jpc3UScez6h7doVRZMkgKUUgKByah)

## 1. Setup and Installations
Run the following cells to install necessary libraries and download NLTK data.

In [1]:
!pip install torch pandas sentence-transformers nltk openpyxl
import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('popular')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lobop\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lobop\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     C:\Users\lobop\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\cmudict.zip.
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     C:\Users\lobop\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\gazetteers.zip.
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     C:\Users\lobop\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\genesis.zip.
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     C:\Users\lobop\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\gutenberg.zip.
[nltk_data]    | Dow

True

## 2. Core Model Engineering
We use **Sentence-BERT (SBERT)** as the base and add a custom Transformer layer for enhanced feature extraction.

In [2]:
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer, util
import os

# Load SBERT model
base_model = SentenceTransformer("all-MiniLM-L6-v2")
original_transformer = base_model._first_module().auto_model

class CustomSBERT(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
        self.extra_layer = nn.TransformerEncoderLayer(d_model=384, nhead=6)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.original_model(input_ids, attention_mask=attention_mask)
        return self.extra_layer(outputs.last_hidden_state)

def load_or_initialize_custom_model(model_path="custom_sbert.pth"):
    model = CustomSBERT(original_transformer)
    if not os.path.exists(model_path):
        torch.save({"model_state_dict": model.state_dict()}, model_path)
        print("✅ Custom SBERT model initialized and saved!")
    else:
        state_dict = torch.load(model_path, map_location=torch.device("cpu"))
        model.load_state_dict(state_dict["model_state_dict"], strict=False)
        model.eval()
        print("✅ Custom SBERT model loaded!")
    return model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 3. Evaluation Logic
The evaluation uses a hybrid score:
- 80% Semantic Similarity (Meaning)
- 20% Keyword Relevance (Specific terminology)

In [8]:
import re
from nltk.corpus import wordnet

def get_synonyms(word):
    return list({lemma.name().replace("_", " ").lower()
                 for syn in wordnet.synsets(word)
                 for lemma in syn.lemmas()})

def check_keyword_match(answer, keyword):
    answer = answer.lower()
    keyword = keyword.lower()
    if keyword in answer: return 1.0
    for synonym in get_synonyms(keyword):
        if synonym in answer: return 0.8
    
    # Fuzzy semantic fallback for keyword
    sim = float(util.cos_sim(base_model.encode(keyword, convert_to_tensor=True),
                             base_model.encode(answer, convert_to_tensor=True))[0][0])
    return 0.6 if sim > 0.7 else 0.0

def compute_scores(student_ans, model_ans, keywords):
    # Semantic Similarity
    emb1 = base_model.encode(student_ans, convert_to_tensor=True)
    emb2 = base_model.encode(model_ans, convert_to_tensor=True)
    sem_score = float(util.cos_sim(emb1, emb2)[0][0])
    
    # Keyword Score
    key_scores = [check_keyword_match(student_ans, kw) for kw in keywords]
    key_score = sum(key_scores) / len(key_scores) if keywords else 0.0
    
    # Weighted Final Score
    final_score = round(min(1.0, 0.8 * sem_score + 0.2 * key_score), 4)
    return sem_score, key_score, final_score

## 4. Run Mini-Evaluation
Test the logic directly with sample input.

In [6]:
sample_model = "Photosynthesis is the process by which green plants use sunlight to synthesize nutrients from carbon dioxide and water."
sample_student = "Plants make food using sunlight, CO2 and water through photosynthesis."
sample_keywords = ["sunlight", "nutrients", "carbon dioxide", "water"]

sem, key, final = compute_scores(sample_student, sample_model, sample_keywords)

print(f"--- Evaluation Results ---")
print(f"Semantic Score: {sem:.4f}")
print(f"Keyword Score: {key:.4f}")
print(f"Final Grade Multiplier: {final:.4f}")

--- Evaluation Results ---
Semantic Score: 0.8072
Keyword Score: 0.7000
Final Grade Multiplier: 0.7858
